# LLM Inference and Analysis from ShareGPT Data

## 1. Setup and Load Data

In [ ]:
import json
import os
import requests
import base64
from PIL import Image
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import time
import re
from IPython.display import display # Added for displaying dataframes

# --- Configuration ---\n",
conversation_file = '../food_conversations.json' # Assumes file is in the parent directory
hpi_api_base = "https://chat.hpi-sci.de/api" # Base URL for HPI API

# Define the models to test
model_identifiers = [
    "mistralai/Pixtral-12B-2409",
    "aihpi/food-waste-vlm"
]

output_results_file_template = 'llm_inference_results_{model_slug}.json' # Template for output files
max_tokens_response = 500 # Max tokens expected in the LLM response JSON
# -- End Configuration --

# Load API Key from .env file (assumed to be in parent directory)
dotenv_path = '.env' # Assuming .env is in the same directory as the notebook now
load_dotenv(dotenv_path=dotenv_path)
api_key = os.getenv('API_KEY')

if not api_key:
    print(f"Error: API_KEY not found in {os.path.abspath(dotenv_path)}")
    # Add manual input or raise error if needed
else:
    print("API Key loaded successfully.")

# Load the ShareGPT conversation data
if os.path.exists(conversation_file):
    try:
        with open(conversation_file, 'r', encoding='utf-8') as f:
            conversations = json.load(f)
        print(f"Loaded {len(conversations)} conversations from {conversation_file}")
    except Exception as e:
        print(f"Error loading {conversation_file}: {e}")
        conversations = [] # Ensure conversations is defined
else:
    print(f"Error: Conversation file not found at {os.path.abspath(conversation_file)}")
    conversations = []

## 2. Helper Functions (API Call, Image Encoding, JSON Parsing)

In [ ]:
def encode_image_to_base64(image_path):
    """Encodes an image file to a Base64 string."""
    if not image_path or not os.path.exists(image_path):
        print(f"Warning: Image path not found or empty: {image_path}")
        return None
    try:
        with Image.open(image_path) as img:
            # Ensure image is in RGB format (common requirement)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffered = io.BytesIO()
            img.save(buffered, format="JPEG")
            return base64.b64encode(buffered.getvalue()).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image {image_path}: {e}")
        return None

def extract_json_from_response(response_text):
    """Extracts the first JSON block from the LLM's response markdown."""
    if not response_text or not isinstance(response_text, str):
        print("Warning: Invalid response text received for JSON extraction.")
        return None
    # Regex to find JSON block enclosed in ```json ... ```
    match = re.search(r'```json\s*({.*?})\s*```', response_text, re.DOTALL)
    if match:
        json_str = match.group(1)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"Error decoding extracted JSON: {e}")
            print(f"Extracted string: {json_str}")
            return None
    else:
        # Fallback: try parsing the whole text if no markdown block found
        try:
            # Attempt to strip leading/trailing whitespace or potential markdown backticks
            cleaned_text = response_text.strip().strip('`')
            return json.loads(cleaned_text)
        except json.JSONDecodeError:
            print("Warning: No JSON block found in response, and response is not valid JSON itself.")
            print(f"Response text: {response_text[:500]}..." ) # Print snippet
            return None

def call_hpi_llm_vision(prompt_text, base64_image, api_key, model_identifier):
    """Calls the HPI multimodal endpoint (assuming OpenAI Vision format).

    Args:
        prompt_text (str): The text prompt.
        base64_image (str or None): Base64 encoded image string or None.
        api_key (str): The API key.
        model_identifier (str): The identifier for the LLM model to use.
    """
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # Construct the payload according to OpenAI Vision API spec
    content_list = [
         {"type": "text", "text": prompt_text}
    ]

    # Add image if available
    if base64_image:
        content_list.append(
             {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                 }
            }
        )

    payload = {
        "model": model_identifier, # Use the passed model identifier
        "messages": [
            {
                "role": "user",
                "content": content_list
            }
        ],
        "max_tokens": max_tokens_response
    }

    # Use the standard chat completions endpoint path
    endpoint = f"{hpi_api_base}/chat/completions"

    try:
        response = requests.post(endpoint, headers=headers, json=payload, timeout=60) # Added timeout
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        response_data = response.json()
        # Extract content from the first choice
        if response_data.get('choices') and len(response_data['choices']) > 0:
            message_content = response_data['choices'][0].get('message', {}).get('content')
            return message_content
        else:
            print(f"Error: No choices found in LLM response for model {model_identifier}.")
            print(f"Response: {response_data}")
            return None

    except requests.exceptions.Timeout:
        print(f"Error: API request timed out for model {model_identifier}.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error calling HPI LLM API for model {model_identifier}: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response status: {e.response.status_code}")
            print(f"Response text: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during API call for model {model_identifier}: {e}")
        return None

## 3. Run Inference Loop

In [ ]:
all_inference_results = {} # Dictionary to store results per model
start_time_all_inference = time.time()

if not conversations:
    print("No conversations loaded. Cannot run inference.")
elif not api_key:
    print("API Key not available. Cannot run inference.")
else:
    total_conversations = len(conversations)
    print(f"Starting inference for {total_conversations} conversations across {len(model_identifiers)} models...")

    for model_id in model_identifiers:
        print(f"\n--- Running Inference for Model: {model_id} ---")
        model_slug = model_id.replace('/', '_') # Create a file-safe name
        model_results = []
        start_time_model_inference = time.time()
        processed_count = 0

        for i, conv in enumerate(conversations):
            print(f"  Processing conversation {i+1}/{total_conversations} for {model_id}...")

            # Extract data from conversation
            try:
                user_message = conv['messages'][0]['content']
                assistant_message = conv['messages'][1]['content'] # Ground Truth
                image_path_rel = conv['images'][0] if conv.get('images') else None
                # Construct absolute path assuming images are relative to conversation file
                if image_path_rel:
                    image_path_abs = os.path.abspath(os.path.join(os.path.dirname(conversation_file), image_path_rel))
                else:
                    image_path_abs = None

                entry_id = os.path.basename(image_path_rel) if image_path_rel else f'conv_{i}'

                # Clean user prompt (remove <image> placeholder for API call)
                prompt_text = user_message.replace('<image>', '').strip()

                # Get ground truth JSON from assistant message
                ground_truth_json = extract_json_from_response(assistant_message)
                if not ground_truth_json:
                    print(f"    Warning: Could not parse ground truth JSON for {entry_id}. Skipping.")
                    continue

            except (IndexError, KeyError, TypeError) as e:
                 print(f"    Error extracting data from conversation {i}: {e}. Skipping.")
                 continue

            # Encode image (using absolute path)
            base64_image = encode_image_to_base64(image_path_abs)
            # Note: If image encoding fails or path is None, base64_image will be None,
            # and the API helper function will send request without image data.

            # Call the LLM for the current model
            print(f"    Calling LLM ({model_id}) for {entry_id}...")
            llm_response_content = call_hpi_llm_vision(prompt_text, base64_image, api_key, model_id)

            # Process response
            result_entry = {
                "id": entry_id,
                "image_path": image_path_rel, # Store relative path
                "ground_truth": ground_truth_json,
                "prediction": None, # Initialize prediction
                "raw_response": llm_response_content # Store raw response for debugging
            }

            if llm_response_content:
                predicted_json = extract_json_from_response(llm_response_content)
                if predicted_json:
                    print(f"    Successfully received and parsed prediction for {entry_id}")
                    result_entry["prediction"] = predicted_json
                else:
                    print(f"    Warning: Failed to parse JSON prediction for {entry_id}")
                    result_entry["prediction"] = {"error": "Failed to parse response"}
            else:
                print(f"    Warning: No valid response received from LLM for {entry_id}")
                result_entry["prediction"] = {"error": "No response from LLM"}

            model_results.append(result_entry)
            processed_count += 1

            # Optional delay between API calls
            # time.sleep(0.5) # Add a small delay if needed

        end_time_model_inference = time.time()
        total_model_inference_time = end_time_model_inference - start_time_model_inference
        print(f"--- Finished inference for {model_id} in {total_model_inference_time:.2f} seconds. ---")
        print(f"--- Successfully processed {processed_count} entries for {model_id}. ---")

        # Store results for this model
        all_inference_results[model_id] = model_results

        # Save results for the current model
        output_file = output_results_file_template.format(model_slug=model_slug)
        if model_results:
            try:
                with open(output_file, 'w', encoding='utf-8') as f:
                    json.dump(model_results, f, indent=4)
                print(f"Inference results for {model_id} saved to {os.path.abspath(output_file)}")
            except Exception as e:
                print(f"Error saving results for {model_id} to {output_file}: {e}")

end_time_all_inference = time.time()
total_inference_time = end_time_all_inference - start_time_all_inference
print(f"\nFinished entire inference loop in {total_inference_time:.2f} seconds.")

## 4. Analyze Results

In [ ]:
def calculate_total_returned(data_json):
    """Calculates total returned weight from prediction or ground truth JSON."""
    total = 0.0
    # Check if the input is a dictionary and contains 'ingredients'
    if not isinstance(data_json, dict) or 'ingredients' not in data_json:
         # Handle cases where prediction failed (e.g., {"error": ...}) or structure is wrong
        if isinstance(data_json, dict) and 'error' in data_json:
            # print(f"  Debug: Encountered error structure in prediction: {data_json}")
            pass # Return None below to indicate failure
        else:
            # print(f"  Debug: Invalid structure or missing 'ingredients': {type(data_json)}")
            pass
        return None # Indicate parsing error or invalid structure

    ingredients = data_json.get('ingredients')
    if not isinstance(ingredients, list):
        # print(f"  Debug: 'ingredients' is not a list: {type(ingredients)}")
        return None # Ingredients should be a list

    for item in ingredients:
        if not isinstance(item, dict):
             # print(f"  Debug: Ingredient item is not a dict: {type(item)}")
             continue # Skip non-dict items

        try:
            # Allow variations in key names ('returned_weight', 'returned weight') case-insensitive
            returned_weight_val = None
            for key in item:
                if key.lower().replace(" ", "_") == 'returned_weight':
                    returned_weight_val = item[key]
                    break

            if returned_weight_val is not None:
                 # Try converting common string formats or handle existing floats/ints
                if isinstance(returned_weight_val, str):
                     # Remove 'g' and convert
                     cleaned_val = returned_weight_val.lower().replace('g', '').strip()
                     returned = float(cleaned_val)
                else:
                     returned = float(returned_weight_val) # Assume float or int
                total += returned
            #else:
                # print(f"  Debug: 'returned_weight' key not found or None in item: {item}")

        except (ValueError, TypeError) as e:
             # print(f"  Debug: Could not convert returned weight to float for item: {item}. Error: {e}")
             pass # Ignore items where returned weight is not a valid number

    return total

# --- Analysis ---
all_analysis_dfs = {}
combined_analysis_data = []

print("\n--- Starting Analysis ---")

# Load results if they weren't computed in this session
if not all_inference_results:
    print("Loading results from files as 'all_inference_results' is empty.")
    for model_id in model_identifiers:
        model_slug = model_id.replace('/', '_')
        output_file = output_results_file_template.format(model_slug=model_slug)
        if os.path.exists(output_file):
            print(f"Loading results for {model_id} from {output_file}")
            try:
                with open(output_file, 'r', encoding='utf-8') as f:
                    all_inference_results[model_id] = json.load(f)
            except Exception as e:
                print(f"Error loading results file {output_file}: {e}")
        else:
            print(f"Warning: Results file not found for {model_id} at {output_file}")

if not all_inference_results:
    print("Error: No inference results available to analyze.")
else:
    print(f"Analyzing results for models: {list(all_inference_results.keys())}")
    # Prepare data for analysis
    for model_id, results in all_inference_results.items():
        print(f"\nProcessing analysis for model: {model_id}")
        analysis_data = []
        valid_entries = 0
        skipped_entries = 0
        for result in results:
            pred_total = calculate_total_returned(result.get('prediction'))
            gt_total = calculate_total_returned(result.get('ground_truth'))

            entry_id = result.get('id', 'unknown_id')

            # Only include entries where both totals could be calculated
            if pred_total is not None and gt_total is not None:
                analysis_data.append({
                    'id': entry_id,
                    'model': model_id, # Add model identifier
                    'predicted_total_returned': pred_total,
                    'actual_total_returned': gt_total
                })
                valid_entries += 1
            else:
                 # print(f"Skipping entry {entry_id} for {model_id} due to missing/invalid prediction ({pred_total}) or ground truth ({gt_total}).")
                 # print(f"  Ground Truth: {result.get('ground_truth')}")
                 # print(f"  Prediction: {result.get('prediction')}")
                 skipped_entries += 1

        print(f"Model {model_id}: Found {valid_entries} valid entries, skipped {skipped_entries} entries.")

        # Create DataFrame for the current model
        if analysis_data:
            model_df = pd.DataFrame(analysis_data)
            all_analysis_dfs[model_id] = model_df
            combined_analysis_data.extend(analysis_data) # Add to combined list
            print(f"Created analysis DataFrame for {model_id} with {len(model_df)} entries.")
            display(model_df.head())
        else:
             print(f"No valid data available for analysis for model {model_id}.")


    # --- Combined Analysis and Comparison ---
    if combined_analysis_data:
        combined_df = pd.DataFrame(combined_analysis_data)
        print(f"\n--- Combined Analysis Results ({len(combined_df)} total valid entries) ---")
        display(combined_df.head())

        # Calculate Metrics per Model
        print("\n--- Evaluation Metrics (Total Returned Weight per Model) ---")
        metrics_summary = []
        for model_id, model_df in all_analysis_dfs.items():
            if not model_df.empty:
                mae = np.mean(np.abs(model_df['predicted_total_returned'] - model_df['actual_total_returned']))
                mse = np.mean((model_df['predicted_total_returned'] - model_df['actual_total_returned'])**2)
                rmse = np.sqrt(mse)

                # Avoid division by zero for MAPE
                model_df_filtered = model_df[model_df['actual_total_returned'] != 0]
                if not model_df_filtered.empty:
                     mape = np.mean(np.abs((model_df_filtered['predicted_total_returned'] - model_df_filtered['actual_total_returned']) / model_df_filtered['actual_total_returned'])) * 100
                else:
                     mape = float('nan') # Assign NaN if no non-zero ground truth values exist

                print(f"\nModel: {model_id}")
                print(f"  MAE:  {mae:.2f}g")
                print(f"  RMSE: {rmse:.2f}g")
                print(f"  MAPE: {mape:.2f}%")
                metrics_summary.append({'Model': model_id, 'MAE (g)': mae, 'RMSE (g)': rmse, 'MAPE (%)': mape})
            else:
                 print(f"\nModel: {model_id} - No data for metrics calculation.")

        if metrics_summary:
             metrics_df = pd.DataFrame(metrics_summary)
             print("\n--- Metrics Summary ---")
             display(metrics_df.round(2))


        # Plotting Comparison
        print("\n--- Plotting Comparison ---")
        num_models = len(all_analysis_dfs)
        if num_models > 0:
            plt.figure(figsize=(8 * num_models, 7)) # Adjust figure size based on number of models

            for i, (model_id, model_df) in enumerate(all_analysis_dfs.items()):
                plt.subplot(1, num_models, i + 1) # Create subplot for each model
                sns.scatterplot(data=model_df, x='actual_total_returned', y='predicted_total_returned', alpha=0.6)

                # Add identity line (perfect prediction)
                if not model_df.empty:
                     # Combine actual and predicted values to find overall min/max for plotting range
                     all_values = pd.concat([model_df['actual_total_returned'], model_df['predicted_total_returned']])
                     min_val = all_values.min()
                     max_val = all_values.max()
                     plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')
                else:
                     plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Perfect Prediction') # Fallback

                plt.title(f'Predicted vs. Actual Total Returned\nModel: {model_id}')
                plt.xlabel('Actual Total Returned (g)')
                plt.ylabel('Predicted Total Returned (g)')
                plt.legend()
                plt.grid(True)
                plt.axis('equal') # Make axes equal scale

            plt.tight_layout() # Adjust layout to prevent overlap
            plt.show()

            # Box plot of errors
            combined_df['error'] = combined_df['predicted_total_returned'] - combined_df['actual_total_returned']
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=combined_df, x='model', y='error')
            plt.title('Distribution of Prediction Errors (Predicted - Actual)')
            plt.xlabel('Model')
            plt.ylabel('Error in Total Returned Weight (g)')
            plt.grid(axis='y')
            plt.axhline(0, color='red', linestyle='--') # Line for zero error
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()

        else:
             print("No dataframes available for plotting.")

    else:
        print("\nNo valid combined data available for analysis after processing results.")